# ECG R-peak exploration — MIT-BIH record 100

A coursework exploration of ECG preprocessing, candidate R-peak detection, and RR-based heart-rate estimation. The notebook uses the first channel of record 100 from the [MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/). It downloads the record when run; no patient data is stored in this repository.

**Scope:** a signal-processing demonstration, not a validated arrhythmia classifier or clinical tool.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, sosfiltfilt, find_peaks


## 1. Download and load a record

Install `requirements.txt` first. The first run needs internet access to fetch record 100 and its annotations from PhysioNet. Subsequent runs reuse the local files.


In [ ]:
DATA_DIR = Path("data/mitdb")
DATA_DIR.mkdir(parents=True, exist_ok=True)
record_path = DATA_DIR / "100"
if not all((DATA_DIR / f"100.{ext}").exists() for ext in ("hea", "dat", "atr")):
    wfdb.dl_database("mitdb", dl_dir=str(DATA_DIR), records=["100"], annotators=["atr"])

record = wfdb.rdrecord(str(record_path))
annotation = wfdb.rdann(str(record_path), "atr")
fs = float(record.fs)
ecg = record.p_signal[:, 0].astype(float)
time = np.arange(ecg.size) / fs
print(f"Record 100: {ecg.size} samples, {fs:g} Hz, channel {record.sig_name[0]}")
print(f"Reference annotations loaded: {len(annotation.sample)} (not used for scoring)")


## 2. Simulate baseline wander

Add a low-frequency sine wave for an illustrative noise experiment. The amplitude is a demonstration setting, not a measured noise level.


In [ ]:
baseline_wander = 0.4 * np.sin(2 * np.pi * 0.1 * time)
noisy_ecg = ecg + baseline_wander
view = time < 10
plt.figure(figsize=(12, 3))
plt.plot(time[view], ecg[view], label="Original", linewidth=1)
plt.plot(time[view], noisy_ecg[view], label="With simulated baseline wander", alpha=0.7)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (mV)")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Filter and find candidate peaks

A 5–15 Hz Butterworth band-pass is followed by differentiation, squaring, and a 150 ms moving average. `find_peaks` applies illustrative thresholds and a 250 ms minimum separation. Candidate positions are then aligned with a nearby maximum in the filtered waveform. These choices are heuristic and have not been benchmarked against the annotations.


In [ ]:
sos = butter(3, [5, 15], btype="bandpass", fs=fs, output="sos")
filtered = sosfiltfilt(sos, noisy_ecg)
energy = np.diff(filtered, prepend=filtered[0]) ** 2
window = max(1, round(0.15 * fs))
integrated = np.convolve(energy, np.ones(window) / window, mode="same")
threshold = 0.35 * np.percentile(integrated, 98)
rough_peaks, _ = find_peaks(
    integrated, height=threshold, distance=max(1, round(0.25 * fs))
)

search_radius = round(0.1 * fs)
aligned = []
for peak in rough_peaks:
    start = max(0, peak - search_radius)
    end = min(len(filtered), peak + search_radius + 1)
    aligned.append(start + np.argmax(filtered[start:end]))
peak_samples = np.unique(aligned).astype(int)
peak_seconds = peak_samples / fs
print(f"Candidate peaks: {len(peak_samples)}")


## 4. Visualize and estimate heart rate

The reported average is `60 / mean(RR intervals)`. It can be misleading when peaks are missed or incorrectly detected.


In [ ]:
rr_seconds = np.diff(peak_seconds)
mean_bpm = 60 / np.mean(rr_seconds) if rr_seconds.size else float("nan")
print(f"Exploratory average heart rate: {mean_bpm:.1f} bpm")

plt.figure(figsize=(12, 3))
plt.plot(time[view], filtered[view], label="Filtered ECG", linewidth=1)
visible = peak_seconds < 10
plt.scatter(peak_seconds[visible], filtered[peak_samples[visible]],
            color="crimson", s=25, label="Candidate R-peaks", zorder=3)
plt.xlabel("Time (s)")
plt.ylabel("Filtered amplitude (mV)")
plt.legend()
plt.tight_layout()
plt.show()


## Limitations and attribution

This notebook demonstrates processing of one channel of one record. Detection accuracy, clinical interpretation, and generalization to other recordings have not been established. Record 100 and its annotations belong to the MIT-BIH Arrhythmia Database, available on PhysioNet under ODC Attribution License v1.0. Cite: Moody GB, Mark RG. *The impact of the MIT-BIH Arrhythmia Database*. IEEE Engineering in Medicine and Biology Magazine 20(3):45–50 (2001).
